# E-commerce Fraud Data EDA

Task 1 notebook for cleaning checks, class imbalance review, IP-to-country enrichment, and fraud pattern exploration.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import FRAUD_DATA, IP_COUNTRY
from src.data_utils import basic_clean, class_distribution, load_fraud_data, load_ip_country
from src.features import add_country_by_ip, engineer_fraud_features


In [2]:
fraud_raw = load_fraud_data(FRAUD_DATA)
ip_ranges = load_ip_country(IP_COUNTRY)
fraud_raw.head()

,user_id,signup_time,purchase_time,purchase_value,device_id,source,browser,sex,age,ip_address,class
0,22058,2015-02-24 22:55:49,2015-04-18 02:47:11,34,QVPSPJUOCKZAR,SEO,Chrome,M,39,7.327584e+08,0
1,333320,2015-06-07 20:39:50,2015-06-08 01:38:54,16,EOGFQPIZPYXFZ,Ads,Chrome,F,53,3.503114e+08,0
2,1359,2015-01-01 18:52:44,2015-01-01 18:52:45,15,YSSKYOSJHPPLJ,SEO,Opera,M,53,2.621474e+09,1
3,150084,2015-04-28 21:13:25,2015-05-04 13:54:50,44,ATGTXKYKUDUQN,SEO,Safari,M,41,3.840542e+09,0
4,221365,2015-07-21 07:09:52,2015-09-09 18:40:53,39,NAUITBZFJKHWW,Ads,Safari,M,45,4.155831e+08,0


In [3]:
cleaning_summary = {
    'rows_before': len(fraud_raw),
    'duplicates': int(fraud_raw.duplicated().sum()),
    'missing_values': fraud_raw.isna().sum().to_dict(),
}
cleaning_summary

{'rows_before': 151112,
 'duplicates': 0,
 'missing_values': {'user_id': 0,
  'signup_time': 0,
  'purchase_time': 0,
  'purchase_value': 0,
  'device_id': 0,
  'source': 0,
  'browser': 0,
  'sex': 0,
  'age': 0,
  'ip_address': 0,
  'class': 0}}

In [4]:
fraud = engineer_fraud_features(add_country_by_ip(basic_clean(fraud_raw), ip_ranges))
class_distribution(fraud, 'class')

,class,count,percentage
0,0,136961,90.635423
1,1,14151,9.364577


In [5]:
fraud[['purchase_value', 'age', 'time_since_signup_hours', 'hour_of_day', 'day_of_week']].describe()

,purchase_value,age,time_since_signup_hours,hour_of_day,day_of_week
count,151112.000000,151112.000000,151112.000000,151112.000000,151112.000000
mean,36.935372,33.140704,1370.008125,11.521593,3.011819
std,18.322762,8.617733,868.406422,6.912474,2.006203
min,9.000000,18.000000,0.000278,0.000000,0.000000
25%,22.000000,27.000000,607.431528,6.000000,1.000000
50%,35.000000,33.000000,1368.429306,12.000000,3.000000
75%,49.000000,39.000000,2123.479028,17.000000,5.000000
max,154.000000,76.000000,2879.992222,23.000000,6.000000


In [6]:
fraud.groupby('class')[['purchase_value', 'age', 'time_since_signup_hours', 'user_transaction_count', 'device_transaction_count']].median()

,purchase_value,age,time_since_signup_hours,user_transaction_count,device_transaction_count
class,,,,,
0,35.0,33.0,1443.030833,1.0,1.0
1,35.0,33.0,0.000278,1.0,8.0


In [7]:
country_fraud = (
    fraud.groupby('country')['class']
    .agg(transactions='count', fraud_rate='mean', fraud_cases='sum')
    .query('transactions >= 20')
    .sort_values(['fraud_rate', 'transactions'], ascending=[False, False])
)
country_fraud.head(15)

,transactions,fraud_rate,fraud_cases
country,,,
Namibia,23,0.434783,10
Sri Lanka,31,0.419355,13
Luxembourg,72,0.388889,28
Ecuador,106,0.264151,28
Tunisia,118,0.262712,31
Peru,119,0.260504,31
Bolivia,53,0.245283,13
Kuwait,90,0.233333,21
Ireland,240,0.229167,55


Document the main EDA takeaways in `reports/interim_report_task1.md`: class imbalance, fraud-heavy countries, suspicious signup-to-purchase timing, and any missing-value decisions.